## Step 1: Environment Setup & Spark Session

In [1]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("SmartCityBusClustering_Storage") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

sc = spark.sparkContext
print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/24 09:02:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/24 09:02:16 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/07/24 09:02:16 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


Spark version: 4.1.1


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 55020)
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
    ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.13/socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.13/socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
    ~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.13/socketserver.py", line 766, in __init__
    self.handle()
    ~~~~~~~~~~~^^
  File "/opt/anaconda3/lib/python3.13/site-packages/pyspark/accumulators.py", line 303, in handle
    poll(accum_updates)
    ~~~~^^^^^^^^^^^^^^^
  File "/o

## Step 2: Project Paths

In [2]:
PROJECT_ROOT = "/Users/aayushbohara/Desktop/smartcity-bus-clustering"
CLEANED_PATH = os.path.join(PROJECT_ROOT, "data/processed/cleaned_dataset.csv")
DB_DIR = os.path.join(PROJECT_ROOT, "data/database")
os.makedirs(DB_DIR, exist_ok=True)
DB_PATH = os.path.join(DB_DIR, "bus_clustering.db")

print("Cleaned data path:", CLEANED_PATH, "| exists:", os.path.exists(CLEANED_PATH))
print("Database will be created at:", DB_PATH)

Cleaned data path: /Users/aayushbohara/Desktop/smartcity-bus-clustering/data/processed/cleaned_dataset.csv | exists: True
Database will be created at: /Users/aayushbohara/Desktop/smartcity-bus-clustering/data/database/bus_clustering.db


## Step 3: Load Cleaned Dataset

In [3]:
df = spark.read.csv(CLEANED_PATH, header=True, inferSchema=True)
print("Row count:", df.count())
df.printSchema()

Row count: 771733
root
 |-- timestamp: timestamp (nullable = true)
 |-- lineRef: string (nullable = true)
 |-- directionRef: string (nullable = true)
 |-- vehicleRef: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- location_source_file: string (nullable = true)
 |-- lineName: string (nullable = true)
 |-- serviceCode: string (nullable = true)
 |-- operator: string (nullable = true)
 |-- nationalOperatorCode: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- vehicleJourneyCode: string (nullable = true)
 |-- departureTime: timestamp (nullable = true)
 |-- journeyPatternRef: string (nullable = true)
 |-- timetable_source_file: string (nullable = true)
 |-- fare_organisation: string (nullable = true)
 |-- common_product_type: string (nullable = true)
 |-- common_tariff_basis: string (nullable = true)
 |-- common_product_name: string (nullable = true)
 |-- fare_product

## Step 4: Build Normalized Dimension Tables

In [4]:
from pyspark.sql.functions import first

# dim_operator: one row per operator
dim_operator = df.select("nationalOperatorCode", "operator") \
    .filter(col("nationalOperatorCode").isNotNull()) \
    .groupBy("nationalOperatorCode") \
    .agg(first("operator").alias("operator_name"))

print("dim_operator rows:", dim_operator.count())
dim_operator.show(truncate=False)

# dim_line: one row per line
dim_line = df.select("lineRef", "lineName", "serviceCode", "nationalOperatorCode", "origin", "destination") \
    .filter(col("lineRef").isNotNull()) \
    .dropDuplicates(["lineRef"])

print("dim_line rows:", dim_line.count())
dim_line.show(5, truncate=False)

# dim_fare: one row per operator's fare summary
dim_fare = df.select("nationalOperatorCode", "fare_organisation", "common_product_type",
                      "common_tariff_basis", "common_product_name", "fare_product_count") \
    .filter(col("nationalOperatorCode").isNotNull() & col("fare_organisation").isNotNull()) \
    .dropDuplicates(["nationalOperatorCode"])

print("dim_fare rows:", dim_fare.count())
dim_fare.show(truncate=False)

dim_operator rows: 4
+--------------------+-------------+
|nationalOperatorCode|operator_name|
+--------------------+-------------+
|BNDB                |Bee Network  |
|BNGN                |Bee Network  |
|BNML                |Bee Network  |
|BNSM                |Bee Network  |
+--------------------+-------------+

dim_line rows: 266
+-------+--------+------------------+--------------------+-----------------------+------------------------------+
|lineRef|lineName|serviceCode       |nationalOperatorCode|origin                 |destination                   |
+-------+--------+------------------+--------------------+-----------------------+------------------------------+
|1      |1       |PC0003681:18010224|BNSM                |Piccadilly Rail Station|Piccadilly Rail Station       |
|10     |10      |PC2021320:18010002|BNGN                |Shudehill Interchange  |Shops                         |
|100    |100     |PC0003681:18030009|BNSM                |Shudehill Interchange  |Warrington 

## Step 5: Build the Fact Table

In [5]:
fact_location_ping = df.select(
    "timestamp", "lineRef", "vehicleRef", "directionRef",
    "latitude", "longitude", "disruption_count",
    "has_timetable_match", "has_fares_match"
)

print("fact_location_ping rows:", fact_location_ping.count())
fact_location_ping.show(5, truncate=False)

fact_location_ping rows: 771733
+-------------------+-------+----------+------------+---------+---------+----------------+-------------------+---------------+
|timestamp          |lineRef|vehicleRef|directionRef|latitude |longitude|disruption_count|has_timetable_match|has_fares_match|
+-------------------+-------+----------+------------+---------+---------+----------------+-------------------+---------------+
|2026-07-22 20:34:28|81     |10602     |outbound    |53.519584|-2.18216 |52              |true               |true           |
|2026-07-22 20:34:28|81     |10602     |outbound    |53.519584|-2.18216 |52              |true               |true           |
|2026-07-22 20:34:28|81     |10602     |outbound    |53.519584|-2.18216 |52              |true               |true           |
|2026-07-22 20:34:28|81     |10602     |outbound    |53.519584|-2.18216 |52              |true               |true           |
|2026-07-22 20:34:28|81     |10602     |outbound    |53.519584|-2.18216 |52    

## Step 6: Convert to Pandas for SQLite Load

In [6]:
dim_operator_pd = dim_operator.toPandas()
dim_line_pd = dim_line.toPandas()
dim_fare_pd = dim_fare.toPandas()

print(dim_operator_pd.shape, dim_line_pd.shape, dim_fare_pd.shape)

(4, 2) (266, 6) (3, 6)


## Step 7: Create SQLite Schema with Foreign Keys (Parameterised, No Raw SQL Injection Risk)

In [7]:
import sqlite3

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS dim_operator")
cursor.execute("DROP TABLE IF EXISTS dim_line")
cursor.execute("DROP TABLE IF EXISTS dim_fare")
cursor.execute("DROP TABLE IF EXISTS fact_location_ping")

cursor.execute("""
CREATE TABLE dim_operator (
    nationalOperatorCode TEXT PRIMARY KEY,
    operator_name TEXT
)
""")

cursor.execute("""
CREATE TABLE dim_line (
    lineRef TEXT PRIMARY KEY,
    lineName TEXT,
    serviceCode TEXT,
    nationalOperatorCode TEXT,
    origin TEXT,
    destination TEXT,
    FOREIGN KEY (nationalOperatorCode) REFERENCES dim_operator(nationalOperatorCode)
)
""")

cursor.execute("""
CREATE TABLE dim_fare (
    nationalOperatorCode TEXT PRIMARY KEY,
    fare_organisation TEXT,
    common_product_type TEXT,
    common_tariff_basis TEXT,
    common_product_name TEXT,
    fare_product_count INTEGER,
    FOREIGN KEY (nationalOperatorCode) REFERENCES dim_operator(nationalOperatorCode)
)
""")

cursor.execute("""
CREATE TABLE fact_location_ping (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    timestamp TEXT,
    lineRef TEXT,
    vehicleRef TEXT,
    directionRef TEXT,
    latitude REAL,
    longitude REAL,
    disruption_count INTEGER,
    has_timetable_match INTEGER,
    has_fares_match INTEGER,
    FOREIGN KEY (lineRef) REFERENCES dim_line(lineRef)
)
""")

conn.commit()
print("Schema created successfully")

Schema created successfully


## Step 8: Insert Dimension Tables (Pandas → SQLite)

In [8]:
dim_operator_pd.to_sql("dim_operator", conn, if_exists="append", index=False)
dim_line_pd.to_sql("dim_line", conn, if_exists="append", index=False)
dim_fare_pd.to_sql("dim_fare", conn, if_exists="append", index=False)

conn.commit()

# verify counts using parameterised queries (no string concatenation)
cursor.execute("SELECT COUNT(*) FROM dim_operator")
print("dim_operator rows in DB:", cursor.fetchone()[0])

cursor.execute("SELECT COUNT(*) FROM dim_line")
print("dim_line rows in DB:", cursor.fetchone()[0])

cursor.execute("SELECT COUNT(*) FROM dim_fare")
print("dim_fare rows in DB:", cursor.fetchone()[0])

dim_operator rows in DB: 4
dim_line rows in DB: 266
dim_fare rows in DB: 3


## Step 9 : Insert Fact Table via Batched Pandas Writes

In [13]:
from pyspark.sql.functions import col as col_, when as when_

fact_for_sql = fact_location_ping \
    .withColumn("has_timetable_match", when_(col_("has_timetable_match"), 1).otherwise(0)) \
    .withColumn("has_fares_match", when_(col_("has_fares_match"), 1).otherwise(0)) \
    .withColumn("timestamp", col_("timestamp").cast("string"))

fact_pd = fact_for_sql.toPandas()
print("Converted to Pandas:", fact_pd.shape)

# batched insert to avoid a single massive transaction
chunk_size = 50000
total_rows = len(fact_pd)

for start in range(0, total_rows, chunk_size):
    chunk = fact_pd.iloc[start:start+chunk_size]
    chunk.to_sql("fact_location_ping", conn, if_exists="append", index=False)
    print(f"Inserted rows {start} to {min(start+chunk_size, total_rows)}")

conn.commit()

cursor.execute("SELECT COUNT(*) FROM fact_location_ping")
print("\nfact_location_ping rows in DB:", cursor.fetchone()[0])

Converted to Pandas: (771733, 9)
Inserted rows 0 to 50000
Inserted rows 50000 to 100000
Inserted rows 100000 to 150000
Inserted rows 150000 to 200000
Inserted rows 200000 to 250000
Inserted rows 250000 to 300000
Inserted rows 300000 to 350000
Inserted rows 350000 to 400000
Inserted rows 400000 to 450000
Inserted rows 450000 to 500000
Inserted rows 500000 to 550000
Inserted rows 550000 to 600000
Inserted rows 600000 to 650000
Inserted rows 650000 to 700000
Inserted rows 700000 to 750000
Inserted rows 750000 to 771733

fact_location_ping rows in DB: 771733


## Step 10: Demonstrate Relationships — Joined Query Across All Tables

In [14]:
# read all 4 tables back from SQLite into Spark DataFrames via temp views for SQL querying
dim_operator_check = spark.createDataFrame(dim_operator_pd)
dim_line_check = spark.createDataFrame(dim_line_pd)
dim_fare_check = spark.createDataFrame(dim_fare_pd)
fact_check = spark.createDataFrame(fact_pd)

dim_operator_check.createOrReplaceTempView("dim_operator")
dim_line_check.createOrReplaceTempView("dim_line")
dim_fare_check.createOrReplaceTempView("dim_fare")
fact_check.createOrReplaceTempView("fact_location_ping")

# example relational query: link fact -> line -> operator -> fare
# demonstrates the "linking timetables with disruptions by Service Code or Operator" requirement
result = spark.sql("""
    SELECT 
        f.lineRef,
        l.lineName,
        l.origin,
        l.destination,
        o.operator_name,
        fr.common_product_name,
        COUNT(*) AS ping_count,
        AVG(f.disruption_count) AS avg_disruption_count
    FROM fact_location_ping f
    JOIN dim_line l ON f.lineRef = l.lineRef
    JOIN dim_operator o ON l.nationalOperatorCode = o.nationalOperatorCode
    LEFT JOIN dim_fare fr ON l.nationalOperatorCode = fr.nationalOperatorCode
    GROUP BY f.lineRef, l.lineName, l.origin, l.destination, o.operator_name, fr.common_product_name
    ORDER BY ping_count DESC
""")

result.show(10, truncate=False)

26/07/24 09:11:41 WARN TaskSetManager: Stage 47 contains a task of very large size (5463 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

+-------+--------+--------------------------+------------------------+-------------+---------------------+----------+--------------------+
|lineRef|lineName|origin                    |destination             |operator_name|common_product_name  |ping_count|avg_disruption_count|
+-------+--------+--------------------------+------------------------+-------------+---------------------+----------+--------------------+
|192    |192     |Chestergate               |Hazel Grove Park & Ride |Bee Network  |1D B+T Z14 CH OP Pass|27020     |51.97224278312361   |
|50     |50      |Huron Basin               |Parrs Wood              |Bee Network  |NULL                 |12660     |52.01263823064771   |
|38     |38      |Piccadilly Gardens        |Lomax Way               |Bee Network  |NULL                 |12300     |51.91382113821138   |
|84     |84      |Piccadilly Gardens        |Uppermill Turning Circle|Bee Network  |1D B+T Z14 CH OP Pass|11890     |51.61648444070648   |
|V1     |V1      |Mancheste

## Step 11: Parameterised Query Example (Injection-Safe Filtering)

In [16]:
target_operator = "BNSM"

# Method 1: DataFrame API filter (already proven safe above)
filtered = dim_line_check.filter(col("nationalOperatorCode") == lit(target_operator))
filtered.show(truncate=False)

# Method 2: Spark SQL with explicit args dict (correct parameter-binding syntax)
filtered_sql = spark.sql(
    "SELECT lineRef, lineName, origin, destination FROM dim_line WHERE nationalOperatorCode = :op",
    args={"op": target_operator}
)
filtered_sql.show(truncate=False)

+-------+--------+------------------+--------------------+---------------------------------+---------------------------------+
|lineRef|lineName|serviceCode       |nationalOperatorCode|origin                           |destination                      |
+-------+--------+------------------+--------------------+---------------------------------+---------------------------------+
|1      |1       |PC0003681:18010224|BNSM                |Piccadilly Rail Station          |Piccadilly Rail Station          |
|100    |100     |PC0003681:18030009|BNSM                |Shudehill Interchange            |Warrington Interchange           |
|11     |11      |PC0003681:18010399|BNSM                |Stockport Interchange            |Altrincham Interchange           |
|112    |112     |PC0003681:18010178|BNSM                |Piccadilly Gardens               |Middleton Bus Station            |
|114    |114     |PC0003681:18010302|BNSM                |Piccadilly Gardens               |Boarshaw Lane      